# NorthForge Finance — End-to-End Workflow Run

Runs the full business workflow through `WorkflowOrchestrator`: Foundry's Trial Balance
pipeline (staging → enrichment → reporting → posting → interface) followed by the GL
import of that pipeline's Interface output — all under a single `WorkflowRun`.

Each section below reads and displays the data actually persisted at that stage, straight
from the domain repositories (`TrialBalanceRepository` for Foundry, `GLRepository` for GL).

## Spark session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('foundry-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/27 20:09:59 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/27 20:09:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7bd74fb3-380a-4f44-80b0-eff227433311;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 72ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#

## Imports and helpers

In [ ]:
from datetime import date

from pyspark.sql import functions as F

from core.logging import configure_logging
from foundry.pipeline import TrialBalancePipeline
from foundry.repository import TrialBalanceRepository
from foundry.config.settings import (
    CSV_TABLE_LOCATIONS,
    POSTGRES_TABLE_LOCATIONS
)

from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from spec import SpecClient
from atlas import AtlasClient
from reference import ReferenceClient

from registry import RegistryClient
from gl import GLClient
from gl.repository import GLRepository

from workflow import WorkflowOrchestrator


configure_logging()


def display_df(df):
    display(df.toPandas())

## Configure clients and build the orchestrator

`TrialBalancePipeline` only knows Foundry processing now — no `RunTracker` — and
`GLClient` only knows GL processing. `WorkflowOrchestrator` is the thin layer that owns
execution/workflow lifecycle across both and coordinates Foundry → GL.

In [3]:
BUSINESS_DT = date(2026, 3, 31)

store = PostgresStore(
    spark,
    table_names = POSTGRES_TABLE_LOCATIONS
)

repository = TrialBalanceRepository(store)

spec = SpecClient.from_db(
    spark,
    transformation_table = 'spec.transformation',
    file_layout_table = 'spec.file_layout'
    
)
atlas = AtlasClient.from_db(
    spark=spark,
    metadata_table='atlas.meta',
    data_table='atlas.data',
)
reference = ReferenceClient.from_db(
    spark = spark,
    fx_rate_table='reference.fx_rate',
    counterparty_table='reference.counterparty',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

pipeline = TrialBalancePipeline(
    business_dt=BUSINESS_DT,
    repository=repository,
    spec=spec,
    atlas=atlas,
    reference=reference,
)

registry = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)
gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry)

orchestrator = WorkflowOrchestrator(
    run_tracker=run_tracker,
    foundry_pipeline=pipeline,
    gl=gl,
)

## Run the full workflow (Foundry → GL)

`run_workflow()` creates a single `WorkflowRun`, executes the Foundry pipeline
(`STAGING → ENRICHMENT → REPORTING → POSTING → INTERFACE`), then runs `GL / IMPORT`
against that same workflow's `FOUNDRY / INTERFACE` output — all as one business workflow.

In [ ]:
workflow_result = orchestrator.run_workflow()

business_dt = pipeline.config.business_dt

# V1 has one producer execution per workflow per output table, so
# workflow_run_id alone is the operational key for every read below.
# gl_run_id (GL's own producer_run_id) is kept only as exact lineage.
workflow_run_id = workflow_result.foundry.identity.workflow_run_id
gl_run_id = workflow_result.gl.producer_run_id

In [ ]:
business_dt = '2026-03-31'
workflow_run_id = '231848ad-fe2b-48b3-9436-f560f7daa1c9'
gl_run_id = '42a38c93-1d6a-4277-8bc2-d592d3b4da32'

## Foundry persistence layers

Each Foundry zone is read straight from its own persisted table, selected by the
`workflow_run_id` of the workflow that produced it (via `TrialBalanceRepository`) — not
by `business_dt`/`batch_id`. `PRODUCER_RUN_ID` remains stamped on every row as the exact
producer execution's lineage.

### Source

In [5]:
src_df = repository.read_source(business_dt=business_dt)

display_df(src_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,SRC_CLIENT_ID,CPTY_REF_ID,SRC_MEASURE_NM,SRC_MEASURE_CCY_CD,SRC_MEASURE_TRANS_AMT,POSTING_MEASURE_CCY_CD
0,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,80000.000000000000,USD
1,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,10000.000000000000,USD
2,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,0E-12,USD
3,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,90000.000000000000,USD
4,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_BACK_VALUED_ADJUSTMENT,USD,0E-12,USD
5,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_ADJUSTED_BALANCE,USD,90000.000000000000,USD
6,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,15000.000000000000,USD
7,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,5000.000000000000,USD
8,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,0E-12,USD
9,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,20000.000000000000,USD


### Staging

In [ ]:
stg_df = repository.read_staging(workflow_run_id)

display_df(stg_df)

### Enrichment

In [ ]:
enr_df = repository.read_enrichment(workflow_run_id)

display_df(enr_df)

### Reporting

In [ ]:
rpt_df = repository.read_reporting(workflow_run_id)

display_df(rpt_df)

### Posting

In [ ]:
pst_df = repository.read_posting(workflow_run_id)

display_df(pst_df)

### Interface

This is the Foundry Interface output — the same rows GL reads as input for `GL / IMPORT`,
selected by `WORKFLOW_RUN_ID` rather than business date/batch.

In [ ]:
int_df = repository.read_interface(workflow_run_id)

display_df(int_df)

## GL persistence layer

`GL / IMPORT` (`gl_run_id`) reads the Interface rows produced by `FOUNDRY / INTERFACE`
above — selected by `workflow_run_id`, since V1 has one Interface producer execution per
workflow — and stamps its own execution identity onto whatever it writes: `gl.posting`
and `gl.rejection` rows carry `PRODUCER_RUN_ID = gl_run_id`, not the Interface producer's
ID. The reads below select by `workflow_run_id` too, not `gl_run_id`.

### GL Posting

In [ ]:
gl_postings = gl.get_postings(workflow_run_id)

display_df(gl_postings)

### GL Rejection

A non-zero rejected count here is a normal business outcome, not an execution failure —
`GL / IMPORT` still completes as `SUCCEEDED` as long as processing itself ran cleanly.

In [ ]:
gl_rejections = gl.get_rejections(workflow_run_id)

display_df(gl_rejections)

In [8]:
print(int_df.columns)
print(gl_postings.columns)

['WORKFLOW_RUN_ID', 'PRODUCER_RUN_ID', 'DATACLASS', 'TRANSACTION_NUMBER', 'LINE_NUMBER', 'ENTITY_CD', 'DEPT_CD', 'BRANCH_CD', 'GL_ACCOUNT', 'SUB_ACCOUNT', 'AFFILIATE_CD', 'PRODUCT_CD', 'BOOK_CD', 'SOURCE_CD', 'CR_DR_IND', 'FOUNDRY_RULE_ID', 'POSTING_ID', 'POSTING_STREAM', 'SRC_RECORD_ID', 'SRC_APP_CD', 'TRANSACTION_CURRENCY', 'TRANSACTION_AMOUNT', 'ACCOUNTED_CURRENCY', 'ACCOUNTED_AMOUNT', 'FX_RATE', 'AS_OF_DATE', 'BUSINESS_DATE']
['GL_POSTING_ID', 'POSTED_AT', 'WORKFLOW_RUN_ID', 'PRODUCER_RUN_ID', 'DATACLASS', 'TRANSACTION_NUMBER', 'LINE_NUMBER', 'FOUNDRY_RULE_ID', 'POSTING_ID', 'POSTING_STREAM', 'SRC_RECORD_ID', 'SRC_APP_CD', 'ENTITY_CD', 'DEPT_CD', 'BRANCH_CD', 'GL_ACCOUNT', 'SUB_ACCOUNT', 'AFFILIATE_CD', 'PRODUCT_CD', 'BOOK_CD', 'SOURCE_CD', 'CR_DR_IND', 'TRANSACTION_CURRENCY', 'TRANSACTION_AMOUNT', 'ACCOUNTED_CURRENCY', 'ACCOUNTED_AMOUNT', 'FX_RATE', 'AS_OF_DATE', 'BUSINESS_DATE']


In [20]:
# reconciliation
recon_keys = [
    'WORKFLOW_RUN_ID',
    'AS_OF_DATE',
    "ENTITY_CD",
    "DEPT_CD",
    "BRANCH_CD",
    "GL_ACCOUNT",
    "SUB_ACCOUNT",
    "AFFILIATE_CD",
    "PRODUCT_CD",
    "BOOK_CD",
    "SOURCE_CD",
    "ACCOUNTED_CURRENCY",
]

interface_recon = (
    int_df
    .groupBy(*recon_keys)
    .agg(
        F.sum("ACCOUNTED_AMOUNT").alias("INTERFACE_BALANCE")
    )
)

gl_recon = (
    gl_postings
    .groupBy(*recon_keys)
    .agg(
        F.sum("ACCOUNTED_AMOUNT").alias("GL_BALANCE")
    )
)

recon = (
    interface_recon
    .join(gl_recon, recon_keys, "full")
    .fillna({
        "INTERFACE_BALANCE": 0,
        "GL_BALANCE": 0,
    })
    .withColumn(
        "DIFFERENCE_AMOUNT",
        F.col("INTERFACE_BALANCE") - F.col("GL_BALANCE"),
    )
)

In [21]:
display_df(interface_recon)

,WORKFLOW_RUN_ID,AS_OF_DATE,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,AFFILIATE_CD,PRODUCT_CD,BOOK_CD,SOURCE_CD,ACCOUNTED_CURRENCY,INTERFACE_BALANCE
0,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,310000,3000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-20000.000000000000
1,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,210000,2000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-65000.000000000000
2,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,410000,4000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-25000.000000000000
3,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,90000.000000000000
4,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,USD,20000.000000000000
5,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,230000,2100,1000,000000,LOCAL_GAAP,NFM_TB,USD,-38250.000000000000
6,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,38250.000000000000


In [22]:
display_df(gl_recon)

,WORKFLOW_RUN_ID,AS_OF_DATE,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,AFFILIATE_CD,PRODUCT_CD,BOOK_CD,SOURCE_CD,ACCOUNTED_CURRENCY,GL_BALANCE
0,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,310000,3000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-20000.000000000000
1,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,210000,2000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-65000.000000000000
2,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,410000,4000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-25000.000000000000
3,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,90000.000000000000
4,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,USD,20000.000000000000
5,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,230000,2100,1000,000000,LOCAL_GAAP,NFM_TB,USD,-38250.000000000000
6,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,38250.000000000000


In [23]:
display_df(recon)

,WORKFLOW_RUN_ID,AS_OF_DATE,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,AFFILIATE_CD,PRODUCT_CD,BOOK_CD,SOURCE_CD,ACCOUNTED_CURRENCY,INTERFACE_BALANCE,GL_BALANCE,DIFFERENCE_AMOUNT
0,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,90000.000000000000,90000.000000000000,0E-11
1,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,210000,2000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-65000.000000000000,-65000.000000000000,0E-11
2,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,USD,20000.000000000000,20000.000000000000,0E-11
3,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,310000,3000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-20000.000000000000,-20000.000000000000,0E-11
4,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,410000,4000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-25000.000000000000,-25000.000000000000,0E-11
5,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,38250.000000000000,38250.000000000000,0E-11
6,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,230000,2100,1000,000000,LOCAL_GAAP,NFM_TB,USD,-38250.000000000000,-38250.000000000000,0E-11


In [13]:
# spark.stop()